# Boundary Indexing & Bulk Generation

Working with boundaries that are too large to enumerate cell by cell.

A resolution-2 cell has **531,438** boundary children at resolution 13 — far too many to
list, let alone draw. This notebook shows the three tools for that situation:

1. `boundary_cell_ids` — the whole set at once, as a NumPy `uint64` array
2. `boundary_cell_at` / `boundary_rank` — reach one cell directly, in O(depth)
3. `boundary_range` — a slice, for streaming or sharding across workers

See the [Boundary Indexing](https://khoshkhah.github.io/h3-boundary/indexing.html) docs for
how the underlying counting rule works.

In [1]:
import time

import h3
import numpy as np
import folium

import h3_boundary as h3b

print("backend:", h3b.get_backend())

parent = h3.latlng_to_cell(37.7759, -122.4180, 2)   # a big cell over California
TARGET = 13
depth = TARGET - h3.get_resolution(parent)

# The boundary size is a closed form — no computation needed
total = 3 ** (depth + 1) - 3
print(f"parent {parent} (res 2) -> res {TARGET}")
print(f"descendants:     {7 ** depth:,}")
print(f"boundary cells:  {total:,}   (3^{depth+1} - 3)")

backend: cpp
parent 822837fffffffff (res 2) -> res 13
descendants:     1,977,326,743
boundary cells:  531,438   (3^12 - 3)


## 1. The whole set at once

`boundary_cell_ids` expands a level at a time using array arithmetic instead of walking
cell by cell, and returns 64-bit indexes rather than hex strings. Both choices matter at
this size.

In [2]:
start = time.perf_counter()
ids = h3b.boundary_cell_ids(parent, target_res=TARGET)
bulk_ms = (time.perf_counter() - start) * 1000

print(f"{len(ids):,} cells in {bulk_ms:.1f} ms  (dtype {ids.dtype}, unordered)")
assert len(ids) == total

# sort=True gives traversal order — the same sequence children_on_boundary_faces returns
start = time.perf_counter()
ordered = h3b.boundary_cell_ids(parent, target_res=TARGET, sort=True)
sorted_ms = (time.perf_counter() - start) * 1000
print(f"sort=True: {sorted_ms:.1f} ms")

print("same cells either way:", set(ids.tolist()) == set(ordered.tolist()))

# The ids feed h3-py's integer API directly — no string round-trip
import h3.api.basic_int as h3i
lat, lng = h3i.cell_to_latlng(int(ids[0]))
print(f"first cell: {lat:.4f}, {lng:.4f}")

531,438 cells in 21.7 ms  (dtype uint64, unordered)
sort=True: 15.3 ms
same cells either way: True
first cell: 39.3194, -122.8729


## 2. One cell, without generating the rest

`boundary_cell_at(parent, target_res, n)` computes the *n*-th boundary cell directly. Cost is
O(depth) — about 15 arithmetic steps — so it does not care that the boundary has half a
million cells. `boundary_rank` is the inverse, and doubles as a membership test.

In [3]:
n = total // 2

# warm up first: the per-state subtree counts are memoized on first use
h3b.boundary_cell_at(parent, target_res=TARGET, n=0)

REPS = 200
start = time.perf_counter()
for i in range(REPS):
    cell = h3b.boundary_cell_at(parent, target_res=TARGET, n=n)
one_ms = (time.perf_counter() - start) * 1000 / REPS

print(f"cell #{n:,} = {cell}   in {one_ms:.4f} ms")
print(f"round trip: boundary_rank -> {h3b.boundary_rank(parent, cell):,}")
print(f"\n{bulk_ms / one_ms:,.0f}x cheaper than generating the whole boundary")

# Interior cells are rejected, which makes rank a membership test too
interior = h3.cell_to_center_child(parent, TARGET)
try:
    h3b.boundary_rank(parent, interior)
except ValueError as e:
    print(f"\ninterior cell correctly rejected: {e}")

cell #265,719 = 8d283451451453f   in 0.0120 ms
round trip: boundary_rank -> 265,719

1,815x cheaper than generating the whole boundary

interior cell correctly rejected: 8d283000000003f is not on the traced boundary of 822837fffffffff


## 3. Sampling a boundary too big to draw

Half a million polygons would kill the browser. Because any cell is directly addressable,
we can draw an evenly spaced sample instead — 300 cells spread around the ring — without
generating the other 531,138.

In [4]:
SAMPLE = 300
picks = [h3b.boundary_cell_at(parent, target_res=TARGET, n=i * total // SAMPLE) for i in range(SAMPLE)]

m = folium.Map(tiles="CartoDB positron")

# the parent cell for context
parent_layer = folium.GeoJson(
    h3b.cell_boundary_to_geojson(parent),
    style_function=lambda x: {"color": "#238636", "weight": 2, "fillOpacity": 0.05},
    name="parent (res 2)",
)
parent_layer.add_to(m)

# the sampled boundary cells, as one FeatureCollection (not one layer each)
folium.GeoJson(
    {"type": "FeatureCollection",
     "features": [h3b.cell_boundary_to_geojson(c) for c in picks]},
    style_function=lambda x: {"color": "#f85149", "weight": 1, "fillOpacity": 0.8},
    name=f"{SAMPLE} sampled boundary cells (res {TARGET})",
).add_to(m)

folium.LayerControl().add_to(m)
m.fit_bounds(parent_layer.get_bounds())
m

The sampled cells trace the parent's outline — including its fractal wiggle, which is why
the boundary grows by a factor of 3 per level rather than √7.

## 4. Slices, and splitting work across workers

`boundary_range` seeks to a position once and then streams forward. Disjoint ranges
reassemble into exactly the traversal's output, so workers need no coordination.

In [5]:
start = time.perf_counter()
window = list(h3b.boundary_range(parent, target_res=TARGET, start=100_000, stop=100_100))
window_ms = (time.perf_counter() - start) * 1000
print(f"100 cells from position 100,000 in {window_ms:.3f} ms")
print(f"first: {window[0]}  (rank {h3b.boundary_rank(parent, window[0]):,})")

# Shard the boundary: worker k of n takes its own contiguous range
WORKERS = 4
bounds = [total * k // WORKERS for k in range(WORKERS + 1)]
shards = [list(h3b.boundary_range(parent, target_res=TARGET, start=bounds[k], stop=bounds[k + 1]))
          for k in range(WORKERS)]

print(f"\n{WORKERS} shards: {[len(s) for s in shards]}")
print("concatenating them reproduces the full boundary:",
      sum(len(s) for s in shards) == total)

100 cells from position 100,000 in 0.239 ms
first: 8d28324926496bf  (rank 100,000)

4 shards: [132859, 132860, 132859, 132860]
concatenating them reproduces the full boundary: True


## Which to use

| You want | Call |
|---|---|
| The whole set, fastest | `boundary_cell_ids(parent, res)` |
| The whole set, in traversal order | `boundary_cell_ids(parent, res, sort=True)` |
| Hex strings | `children_on_boundary_faces(parent, res)` |
| One cell, or a sample | `boundary_cell_at(parent, target_res, n)` |
| A cell's position / membership | `boundary_rank(parent, cell)` |
| A slice, or one worker's share | `boundary_range(parent, res, lo, hi)` |
| Just the count | `3**(depth+1) - 3` |